In [1]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 155.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.10.1
    Uninstalling transformers-5.10.1:
      Successfully uninstalled transformers-5.10.1


In [2]:
!pip install peft bitsandbytes accelerate
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 111.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
notebook_name = 'summarizer'

**Huggingface login**

In [5]:
from huggingface_hub import login

login()

**Downloading dataset**

In [6]:
from datasets import load_dataset

dataset = load_dataset("OpenHust/vietnamese-summarization")

README.md:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

Kmeans_1024_new.csv:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

Kmeans_512_new.csv:   0%|          | 0.00/25.0M [00:00<?, ?B/s]

Kmeans_512_token_new.csv:   0%|          | 0.00/29.1M [00:00<?, ?B/s]

bio_medicine.csv:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

herding_512_bio_medicine.csv:   0%|          | 0.00/25.1M [00:00<?, ?B/s]

herding_bio_medicine.csv:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

herding_prompt_512_bio_medicine.csv:   0%|          | 0.00/29.1M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

**Data processing**

In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Document', 'Summary', 'Dataset'],
        num_rows: 74564
    })
})

In [8]:
dataset = dataset['train'].select(range(10000))
dataset

Dataset({
    features: ['Unnamed: 0', 'Document', 'Summary', 'Dataset'],
    num_rows: 10000
})

In [9]:
split_dataset = dataset.train_test_split(test_size=0.2)

train_set = split_dataset['train']
test_set = split_dataset['test']

In [11]:
train_set
test_set

Dataset({
    features: ['Unnamed: 0', 'Document', 'Summary', 'Dataset'],
    num_rows: 2000
})

In [10]:
train_set[0]

{'Unnamed: 0': 9300,
 'Document': 'Nếu quyết định sử dụng biện pháp ngừa thai nội tiết tố dưới dạng thuốc viên, miếng dán, hoặc vòng âm đạo, bạn cần trao đổi với bác sĩ trước. Bác sĩ sẽ đưa ra một vài lựa chọn và sau đó chỉ định phương pháp phù hợp với sức khỏe và lối sống của bạn.  Bạn cần phải được kê toa trước khi bắt đầu sử dụng thuốc, miếng dán hoặc vòng âm đạo ngừa thai.  Thuốc ngừa thai có nhiều loại khác nhau từ 21 cho đến 365 ngày cũng như kết hợp đa dạng estrogen và progestin. Trong thời gian không dùng thuốc, bạn sẽ có kinh nguyệt.  Vòng âm đạo là vòng nhựa dẻo được đưa vào âm đạo trong vòng 21 ngày. Chúng sản sinh nội tiết tố thấm vào mô âm đạo và chuyển sang toàn bộ cơ thể. Sau 21 ngày, kinh nguyệt sẽ xuất hiện.  Miếng dán tránh thai được dán lên da trong ba tuần liên tiếp. Bạn sẽ có kinh nguyệt ở tuần thứ tư và sau đó sử dụng miếng dán mới. Chúng tiết ra estrogen và progestin thâm nhập vào cơ thể qua bề mặt da. Trước khi bắt đầu sử dụng biện pháp tránh thai nội tiết tố, b

### Trim dataset

In [ ]:
def normalize_text(row):
    # Only handle lowercase normalization
    row['Document'] = str(row['Document']).lower()
    row['Summary'] = str(row['Summary']).lower()
    return row

def add_special_tokens(row):
    # Qwen follows the ChatML format:
    # <|im_start|>user
    # {Document}<|im_end|>
    # <|im_start|>assistant
    # {Summary}<|im_end|>

    formatted_input = (
        f"<|im_start|>\n"
        f"{row['Document']}<|im_end|>\n"
        f"<|im_start|>\n"
        f"{row['Summary']}<|im_end|>"
    )

    row['Processed_Input'] = formatted_input
    return row

def process(row):
    normalized_row = normalize_text(row)
    return add_special_tokens(normalized_row)

train_set = train_set.map(process)
test_set = test_set.map(process)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
print("Special tokens: ", tokenizer.special_tokens_map)
sep_token_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
print(f"Sep token is {sep_token_id}")

Special tokens:  {'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'audio_bos_token': '<|audio_start|>', 'audio_eos_token': '<|audio_end|>', 'audio_token': '<|audio_pad|>', 'image_token': '<|image_pad|>', 'video_token': '<|video_pad|>', 'vision_bos_token': '<|vision_start|>', 'vision_eos_token': '<|vision_end|>'}
Sep token is 248045


In [ ]:
def tokenize_function(batch):
    # Tokenize the processed input
    model_inputs = tokenizer(
        batch["Processed_Input"],
        truncation=True,
        max_length=1542,
        padding='max_length'
    )

    labels = []

    for input_ids in model_inputs["input_ids"]:
        label = list(input_ids)
        try:
            # Find the index of the first separation token
            indices = [i for i, val in enumerate(label) if val == sep_token_id]
            turn_model_idx = indices[-1]
            # Set all tokens up to the first separation to -100 (loss function ignore)
            for i in range(turn_model_idx + 1):
                label[i] = -100
        except ValueError as e:
            print(e)

        labels.append(label)

    model_inputs["labels"] = labels
    return model_inputs


In [ ]:
tokenized_train = train_set.map(tokenize_function, batched=True, remove_columns=train_set.column_names)
tokenized_test = test_set.map(tokenize_function, batched=True, remove_columns=test_set.column_names)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

**Fine-tuning**

In [ ]:
from transformers import TrainingArguments
import os

training_args = TrainingArguments(
    output_dir=os.path.join("/content/drive/MyDrive/ML", notebook_name),
    num_train_epochs=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=32,
    gradient_checkpointing=True,
    bf16=True,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    optim="adamw_bnb_8bit"
)

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 248044}.


Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 11.41 GiB. GPU 0 has a total capacity of 39.49 GiB of which 3.69 GiB is free. Including non-PyTorch memory, this process has 35.79 GiB memory in use. Of the allocated memory 33.92 GiB is allocated by PyTorch, and 1.36 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)